# Frequency Guard - Colab Deployment
Deploy Frequency Guard web application using ngrok tunnel

## 1. Install Node.js

In [ ]:
%%bash
# Install Node.js 20.x
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt-get install -y nodejs
node --version
npm --version

## 2. Upload Project Files
Upload your project as a ZIP file or clone from GitHub

In [ ]:
import os
from google.colab import files
import zipfile

# Option 1: Upload ZIP file
print("Please upload your frequency-guard.zip file")
uploaded = files.upload()

# Extract the uploaded ZIP
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print(f"Extracted {filename}")

# List extracted contents
!ls -la /content/

In [ ]:
# Option 2: Clone from GitHub (uncomment and modify if using GitHub)
# !git clone https://github.com/YOUR_USERNAME/frequency-guard.git /content/frequency-guard

## 3. Install Project Dependencies

In [ ]:
%%bash
# Change to project directory (adjust path if needed)
cd /content/frequency-guard

# Install dependencies
npm install

## 4. Setup ngrok

In [ ]:
%%bash
# Download and install ngrok
wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
tar -xzf ngrok-v3-stable-linux-amd64.tgz
chmod +x ngrok
mv ngrok /usr/local/bin/
ngrok version

In [ ]:
# Add your ngrok authtoken (get it from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # Replace with your token

!ngrok authtoken {NGROK_AUTH_TOKEN}

## 5. Start Development Server and ngrok Tunnel

In [ ]:
import subprocess
import time
import requests
from IPython.display import display, HTML

# Start Vite dev server in background
print("Starting Vite development server...")
vite_process = subprocess.Popen(
    ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173"],
    cwd="/content/frequency-guard",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
time.sleep(10)

# Start ngrok tunnel
print("Starting ngrok tunnel...")
ngrok_process = subprocess.Popen(
    ["ngrok", "http", "5173"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for ngrok to start
time.sleep(5)

# Get ngrok public URL
try:
    response = requests.get("http://localhost:4040/api/tunnels")
    tunnels = response.json()["tunnels"]
    public_url = tunnels[0]["public_url"]
    
    print("\n" + "="*60)
    print("🎉 Frequency Guard is now live!")
    print("="*60)
    print(f"\n🔗 Public URL: {public_url}")
    print("\n📝 Keep this cell running to maintain the tunnel")
    print("❌ Stop the cell to shut down the server\n")
    
    # Display clickable link
    display(HTML(f'<h2><a href="{public_url}" target="_blank" style="color: #2196F3;">Click here to open Frequency Guard →</a></h2>'))
    
    # Keep running
    print("Server is running... Press stop button to terminate.")
    vite_process.wait()
    
except Exception as e:
    print(f"Error: {e}")
    print("Trying to get ngrok URL again...")
    time.sleep(3)
    try:
        response = requests.get("http://localhost:4040/api/tunnels")
        tunnels = response.json()["tunnels"]
        public_url = tunnels[0]["public_url"]
        print(f"\n🔗 Public URL: {public_url}\n")
        display(HTML(f'<h2><a href="{public_url}" target="_blank" style="color: #2196F3;">Click here to open Frequency Guard →</a></h2>'))
        vite_process.wait()
    except:
        print("Could not retrieve ngrok URL. Check ngrok dashboard at: https://dashboard.ngrok.com/")

## 6. Check Server Logs (Optional)

In [ ]:
# View recent server logs
!tail -n 50 /content/frequency-guard/npm-debug.log 2>/dev/null || echo "No error logs found"

## Troubleshooting

If the server doesn't start:
1. Make sure the project folder name is correct (`frequency-guard`)
2. Check if all dependencies installed successfully
3. Verify ngrok authtoken is correct
4. Try restarting the runtime and running cells again

**Note:** Free ngrok tunnels expire after 2 hours. You'll need to restart cell 5 to get a new URL.